# PoC — a terceira coordenada ajuda?

**Pergunta:** o pipeline descarta o `z` dos landmarks (`dados.py`, `arr[:, :, :2]`).
Essa decisão foi **herdada da PoC do DTW**, onde foi medida (64,0% → 68,7% ao
desligar o z, `PoC/config.yaml`), e **nunca testada** na ResNet nem no ST-GCN.

**Por que a medição do DTW não transfere.** O DTW soma distâncias cruas: um canal com
escala incoerente corrompe a métrica sem defesa. ResNet e GCN têm peso aprendido e podem
ponderar ou ignorar o canal. A conclusão do DTW não vale para eles em nenhuma das duas
direções — nem "o z piora", nem "a rede aprende a usar".

**Por que virou urgente.** `docs/extracao-landmarks-plano.md` §3 decidiu que o app vai
extrair 3 canais. O modelo é construído com `canais_ent=2`. Isso não é acurácia pior: é
**erro de shape** na integração. Alguém tem de ceder, e a decisão pertence ao treino,
porque é aqui que dá para medir.

---

## O que este notebook mede

Três variantes, **mesma semente**, mesmo tudo o mais:

| Variante | Coordenadas | Papel |
|---|---|---|
| A | x, y | **controle** — não é o 93,4% de antes, é uma execução nova nas mesmas condições |
| B | x, y, z | o z como está hoje no `.npy` |
| C | x, y, z recentrado | o z das mãos devolvido ao referencial do punho |

**Por que C existe.** Em `extract.py`, a origem subtraída de todos os pontos é o meio dos
ombros — um vetor de 3 dimensões. Para a pose, tudo bem. Para as mãos, não: o z de mão do
MediaPipe **já vem relativo ao punho**, e subtrair dele o z do ombro mistura dois
referenciais. Medido em 150 clipes, isso domina o valor típico: a mediana de |z| cai de
**0,74 para 0,18** ao recentrar. Rodar só A vs B mediria o defeito, não a informação — e
provavelmente reproduziria o resultado do DTW pelo motivo errado.

**Por que a semente fixa.** A mesma configuração oscila ~1,7 pp entre execuções. Com um
efeito esperado de poucos pontos, rodar cada variante uma vez sem semente mede ruído.
`--semente` faz as três partirem dos mesmos pesos e da mesma augmentação; o que sobra na
diferença é a representação.

> ⚠️ Nada aqui mede robustez a ângulo de câmera, que é a motivação de fundo para querer
> profundidade. Todas as bases são estúdio frontal. Um ganho aqui é ganho em vídeo
> frontal — não licença para afirmar que o z resolve o problema dos óculos.

---

## Onde rodar

Roda em **Kaggle** ou **Colab** — o notebook detecta e se adapta. A diferença que importa
é onde os resultados sobrevivem se a conexão cair:

| | Background | Resultado sobrevive à queda |
|---|---|---|
| **Kaggle** (*Save & Run All*) | sim | sim, vira output da versão |
| Colab grátis | não | só porque este notebook grava `EXP` no Drive |

No Kaggle: telefone verificado, `landmarks-minds.tar.gz` como dataset privado de slug
**`libras-landmarks`**, Accelerator **GPU** e Internet **On** (a primeira célula clona do
GitHub). Passo a passo em [`docs/poc-tres-coordenadas.md`](../../docs/poc-tres-coordenadas.md).

⚠️ *Save & Run All* roda o notebook inteiro, **incluindo a célula opcional do ST-GCN**
(mais de 1h30). Para só o experimento principal, esvazie `GCN_VARIANTES` e `RESNET_EXTRA`.


## 1. Ambiente e código

In [ ]:
import os, pathlib, shutil, subprocess, sys

EM_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
EM_KAGGLE = os.path.exists("/kaggle/working")
BASE = pathlib.Path("/content" if EM_COLAB else "/kaggle/working" if EM_KAGGLE else ".")
print("ambiente:", "Colab" if EM_COLAB else "Kaggle" if EM_KAGGLE else "local", "| base:", BASE)

URL = "https://github.com/Heitorvazeg/libras-livre-ai-glasses-brasil.git"
BRANCH = "poc/tres-coordenadas"   # PoC das 3 coordenadas
REPO = BASE / "libras-livre-ai-glasses-brasil"
TREINO = REPO / "computer-vision-model" / "treino"

def git(*args, repo=None):
    cmd = ["git"] + (["-C", str(repo)] if repo else []) + list(args)
    return subprocess.run(cmd, capture_output=True, text=True)

# Esta célula deixa o código SEMPRE atual, em três situações diferentes:
#   1. não há clone            -> clona a branch certa
#   2. há clone, branch errada -> descarta e reclona (foi o que aconteceu quando
#                                 o clone veio da default e não tinha treino/)
#   3. há clone, branch certa  -> ATUALIZA. Sem isso, reabrir o notebook num
#                                 runtime que ainda vive reaproveita código velho
#                                 e as correções recém-publicadas não chegam.
if REPO.exists() and (git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO).stdout.strip() != BRANCH
                      or not TREINO.is_dir()):
    print("clone existente está na branch errada ou incompleto — refazendo")
    shutil.rmtree(REPO)

if REPO.exists():
    antes = git("rev-parse", "--short", "HEAD", repo=REPO).stdout.strip()
    git("fetch", "--depth", "1", "origin", BRANCH, repo=REPO)
    git("reset", "--hard", f"origin/{BRANCH}", repo=REPO)
    depois = git("rev-parse", "--short", "HEAD", repo=REPO).stdout.strip()
    print(f"repo atualizado: {antes} -> {depois}" if antes != depois else
          f"repo já estava atual ({depois})")
else:
    # --branch no clone: sem isso vem a default (main), que não tem treino/.
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL,
                            str(REPO)], capture_output=True, text=True)
    if clone.returncode:
        dica = ("\n\nNO KAGGLE: internet vem DESLIGADA por padrão. Abra o painel da\n"
                "direita -> Notebook options -> Internet: On (exige telefone\n"
                "verificado na conta). Sem isso o clone não tem como funcionar."
                if EM_KAGGLE else "")
        raise SystemExit(f"git clone falhou:\n{clone.stderr.strip()}{dica}")

assert TREINO.is_dir(), f"{TREINO} não existe mesmo após o clone"
print("branch:", git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO).stdout.strip())
print("último commit:", git("log", "-1", "--pretty=%h %s", repo=REPO).stdout.strip())
print("código em:", TREINO)


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Ative a GPU antes de continuar; não executar treino pesado em CPU.")
print("GPU:", torch.cuda.get_device_name(0))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "scipy"], check=True)

## 2. Pacote privado de landmarks (MINDS)

In [ ]:
import json
import tarfile
import tempfile

DESTINO = (REPO / "computer-vision-model" / "PoC" / "data").resolve()
DESTINO.mkdir(parents=True, exist_ok=True)
# Só MINDS: esta PoC não pré-treina, compara representações no LOSO. Menos
# upload, menos coisa para dar errado, e V-LIBRASIL não entra na avaliação.
PACOTES = {"landmarks-minds.tar.gz": "landmarks"}

if EM_COLAB:
    from google.colab import files
    print("Selecione o pacote privado: " + ", ".join(PACOTES))
    enviados = files.upload()
    if set(enviados) != set(PACOTES):
        raise ValueError("Envie exatamente o pacote esperado, com o nome indicado.")
    origens = {nome: pathlib.Path(nome).resolve() for nome in PACOTES}
    del enviados  # libera as cópias dos pacotes em memória
else:
    # Ajuste para suas entradas PRIVADAS no Kaggle ou caminhos locais autorizados.
    raiz_pacotes = pathlib.Path("/kaggle/input/libras-landmarks" if EM_KAGGLE else "~").expanduser()
    origens = {nome: raiz_pacotes / nome for nome in PACOTES}

if not hasattr(tarfile, "data_filter"):
    raise RuntimeError("Atualize o Python: extração exige filter='data', sem fallback.")
for nome, pasta in PACOTES.items():
    if not origens[nome].is_file():
        raise FileNotFoundError(origens[nome])
    alvo = DESTINO / pasta
    if alvo.is_symlink() or (alvo.exists() and (not alvo.is_dir() or any(
        p.name != ".gitkeep" or not p.is_file() or p.is_symlink() for p in alvo.iterdir()
    ))):
        raise RuntimeError(f"Destino já preenchido: {alvo}. Use um runtime/diretório limpo.")

# Valida os dois pacotes numa área temporária antes de instalar qualquer um.
with tempfile.TemporaryDirectory(dir=DESTINO) as temporario:
    staging = pathlib.Path(temporario)
    for nome, pasta in PACOTES.items():
        with tarfile.open(origens[nome], "r:gz") as tar:
            membros = tar.getmembers()
            vistos = set()
            for membro in membros:
                caminho = pathlib.PurePosixPath(membro.name)
                if caminho.is_absolute() or ".." in caminho.parts or not caminho.parts or caminho.parts[0] != pasta:
                    raise ValueError(f"Caminho inesperado no pacote {nome}: {membro.name}")
                if caminho in vistos:
                    raise ValueError(f"Membro duplicado: {membro.name}")
                vistos.add(caminho)
                if membro.isdir() and len(caminho.parts) == 1:
                    continue
                if not (membro.isfile() and len(caminho.parts) == 2
                        and caminho.name.endswith((".npy", ".npy.proveniencia.json"))):
                    raise ValueError(f"Só landmarks/sidecars regulares são permitidos: {membro.name}")
            tar.extractall(staging, members=membros, filter="data")

        npys = sorted((staging / pasta).glob("*.npy"))
        prefixo = "pessoaM" if pasta == "landmarks" else "pessoaV"
        if not npys or any(not p.name.startswith(prefixo) for p in npys):
            raise ValueError(f"Pacote vazio ou fonte incorreta: {nome}")
        for npy in npys:
            sidecar = npy.with_name(npy.name + ".proveniencia.json")
            if pasta == "landmarks-pretreino" or sidecar.exists():
                if not sidecar.is_file():
                    raise FileNotFoundError(f"Proveniência obrigatória ausente: {sidecar.name}")
                meta = json.loads(sidecar.read_text(encoding="utf-8"))
                if not isinstance(meta, dict) or not meta:
                    raise ValueError(f"Sidecar deve conter um objeto JSON não vazio: {sidecar.name}")
        for sidecar in (staging / pasta).glob("*.npy.proveniencia.json"):
            if not sidecar.with_name(sidecar.name.removesuffix(".proveniencia.json")).is_file():
                raise ValueError(f"Sidecar sem landmark correspondente: {sidecar.name}")
        pessoas = sorted({p.name.split("_")[0] for p in npys})
        if pasta == "landmarks-pretreino" and "pessoaV03" not in pessoas:
            raise ValueError("V03 não está no corpus; não é possível reservar a validação explícita.")
        print(f"{pasta}: {len(npys)} clipes | pessoas: {', '.join(pessoas)}")

    for pasta in PACOTES.values():
        alvo = DESTINO / pasta
        alvo.mkdir(exist_ok=True)
        # Preserva o .gitkeep do clone limpo; nunca remove dados de uma execução antiga.
        for item in (staging / pasta).iterdir():
            shutil.move(str(item), str(alvo / item.name))


## 3. As três variantes

A primeira célula prepara e valida; a segunda treina. Estão separadas de propósito: no
Colab a montagem do Drive é interativa, e precisa acontecer **antes** da rodada longa —
se ficar depois, ela só pede autorização quando o treino terminar, o que não serve para
quem caiu no meio.

Cada variante roda o LOSO completo (8 rodadas) com a mesma semente. Em T4, minutos cada.


In [ ]:
from datetime import datetime
from uuid import uuid4

TREINO = TREINO.resolve()
nome_exp = "poc3d-" + datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]

# ONDE OS RESULTADOS SOBREVIVEM, por ambiente. Decidido AQUI, antes de treinar,
# e não copiado depois: copiar no fim só protege quem chegou ao fim.
#
#   Kaggle  /kaggle/working é o output da versão. Com "Save & Run All" o notebook
#           roda na infra deles; fechar o navegador não perde nada.
#   Colab   /content morre com o runtime, e runtime ocioso é reciclado. Então
#           gravamos DENTRO do Drive: se a sessão cair no meio, o que já rodou
#           está salvo.
if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # interativo: por isso vem antes do treino
    RAIZ_SAIDA = pathlib.Path("/content/drive/MyDrive/libras-poc3d")
else:
    RAIZ_SAIDA = BASE.resolve() / "experimentos-privados"

EXP = RAIZ_SAIDA / nome_exp
EXP.mkdir(parents=True, exist_ok=False)
print("resultados em:", EXP)
if EM_KAGGLE:
    print("Kaggle: isto vira output desta versão — nada a baixar durante a execução.")

SEMENTE = "20260916"   # o mesmo valor nas três: é o que torna a comparação pareada
ARQUITETURA = "resnet" # a ResNet é o modelo do MVP; o GCN fica na célula opcional

BASE_ARGS = [
    "--arquitetura", ARQUITETURA, "--fontes", "minds", "--dispositivo", "cuda",
    "--epocas", "30", "--lr", "1e-4", "--batch", "64", "--agendador", "nenhum",
    "--folds", "0", "--semente", SEMENTE,
]

VARIANTES = {
    "A-xy":            [],
    "B-xyz":           ["--com-z"],
    "C-xyz-recentrado": ["--com-z", "--z-recentrado"],
}

# Encanamento antes de gastar GPU: se o selftest quebrar, as três rodariam errado.
subprocess.run([sys.executable, "selftest.py"], cwd=TREINO, check=True)
print("selftest OK — pode treinar")


In [ ]:
for nome, extra in VARIANTES.items():
    print("\n" + "=" * 70 + f"\n### {nome}\n" + "=" * 70, flush=True)
    subprocess.run(
        [sys.executable, "treinar.py", *BASE_ARGS, *extra, "--saida", str(EXP / nome)],
        cwd=TREINO, check=True)
print("\nSaídas privadas:", EXP)


## 4. Comparação

Lê os `relatorio.md` gerados e monta a tabela fold a fold. O delta é sempre contra a
variante A (o controle desta mesma execução), nunca contra o 93,4% histórico — aquele
número veio de outra máquina, outro batch e sem semente.


In [ ]:
import re

def ler(nome):
    txt = (EXP / nome / "relatorio.md").read_text(encoding="utf-8")
    folds = dict(re.findall(r"^\| (M\d+) \| ([\d.]+)%", txt, re.M))
    media = float(re.search(r"média = ([\d.]+)%", txt).group(1))
    return {k: float(v) for k, v in folds.items()}, media

# Lê o que EXISTIR: a célula opcional pode não ter rodado, e comparar contra
# uma variante ausente quebraria a leitura das que rodaram.
GRUPOS = [("ResNet", list(VARIANTES)
           + [n for n in ("A-xy-sem-imput",) if (EXP / n / "relatorio.md").is_file()]),
          ("ST-GCN", [n for n in ("G-xy", "G-xy-sem-imput", "G-xyz-rec", "G-xy-ossos")
                      if (EXP / n / "relatorio.md").is_file()])]
res = {}
for titulo, nomes in GRUPOS:
    if not nomes:
        continue
    for n in nomes:
        res[n] = ler(n)
    ctrl_nome = nomes[0]
    ctrl = res[ctrl_nome][0]
    pessoas = sorted(ctrl)
    print(f"\n### {titulo} — controle: {ctrl_nome}")
    print(f"{'fold':<7}" + "".join(f"{n:>22}" for n in nomes))
    print("-" * (7 + 22 * len(nomes)))
    for p in pessoas:
        linha = f"{p:<7}"
        for n in nomes:
            a = res[n][0][p]
            linha += f"{a:>13.1f}%{'' if n == ctrl_nome else f' ({a - ctrl[p]:+.1f})':>8}"
        print(linha)
    print("-" * (7 + 22 * len(nomes)))
    linha = f"{'MÉDIA':<7}"
    positivos = {}
    for n in nomes:
        m = res[n][1]
        linha += f"{m:>13.1f}%{'' if n == ctrl_nome else f' ({m - res[ctrl_nome][1]:+.1f})':>8}"
        if n != ctrl_nome:
            d = [res[n][0][p] - ctrl[p] for p in pessoas]
            positivos[n] = (sum(x > 0 for x in d), sum(x < 0 for x in d), len(d))
    print(linha)
    # A consistência entre folds diz mais que a média: 7 de 8 subindo é sinal,
    # +6 num fold e -4 noutro é ruído com cara de resultado.
    for n, (pos, neg, tot) in positivos.items():
        print(f"  {n}: {pos}/{tot} folds acima do controle, {neg} abaixo")

print("""
COMO LER. A variância de execução para execução é ~1,7 pp. Semente fixa reduz isso,
mas não a zero: GPU não é bit-a-bit determinística. Trate diferença de média abaixo
de ~1,5 pp como empate, e olhe a CONSISTÊNCIA entre folds — um ganho real aparece na
maioria dos 8, não como +6 num fold e -4 noutro.

SE B e C empatarem com A: o z não carrega informação que estes modelos aproveitem em
vídeo frontal. O app deve extrair 2 canais, e `extracao-landmarks-plano.md` §3 muda.

SE C ganhar e B não: o problema era a normalização, não o z. Corrigir `extract.py`
vira trabalho real, e o app deve extrair 3 canais JÁ RECENTRADOS.

SE B e C ganharem igual: o offset do ombro não atrapalhava tanto quanto a medição de
mediana sugeria; usar o z cru basta.
""")


## 5. Opcional — ST-GCN e a imputação pareada

Duas perguntas que sobram, ambas baratas em GPU e caras em CPU:

**(a) A imputação de lacunas de mão ajuda o ST-GCN?** Ela mora em `dados.py`, na leitura,
então vale para as duas arquiteturas — mas só foi medida na ResNet. Há motivo para o
efeito ser diferente no grafo: uma mão zerada vira 21 nós teleportados para a origem, e a
convolução espacial propaga isso para os vizinhos pela aresta punho↔mão. O dano pode ser
maior ali, e o ganho também.

**(b) Quanto a imputação vale na ResNet, medido de forma pareada?** O número que temos
(93,4% → 95,1%) veio de duas execuções **sem semente fixa** — comparação não pareada, com
ruído de inicialização em cada fold. Rodar `--sem-imputacao` com a MESMA semente da
variante A dá o delta limpo.

> ⏱️ O ST-GCN treina do zero (120 épocas contra 30 da ResNet). Em T4 são ~20-30 min por
> variante; as quatro do GCN passam de 1h30. O Colab derruba runtime ocioso — não feche a
> aba. Se o tempo apertar, rode primeiro `G-xy` e `G-xy-sem-imput`, que respondem (a).


In [ ]:
# ST-GCN: 120 épocas, lr 1e-3, cosseno — orçamento de treino DO ZERO. Usar a config
# de fine-tuning aqui foi o erro que fez o GCN "perder" por 49 pontos (CONTEXTO.md §6).
GCN_ARGS = [a if a != "resnet" else "gcn" for a in BASE_ARGS]
GCN_ARGS[GCN_ARGS.index("--epocas") + 1] = "120"
GCN_ARGS[GCN_ARGS.index("--lr") + 1] = "1e-3"
GCN_ARGS[GCN_ARGS.index("--agendador") + 1] = "cosseno"

# G-xy é o controle das outras três: mesma semente, só a variável em teste muda.
GCN_VARIANTES = {
    "G-xy":             [],
    "G-xy-sem-imput":   ["--sem-imputacao"],          # (a) imputação no grafo
    "G-xyz-rec":        ["--com-z", "--z-recentrado"],
    "G-xy-ossos":       ["--ossos"],
}

# (b) A MESMA semente da variante A do experimento principal, para o par ficar limpo.
RESNET_EXTRA = {"A-xy-sem-imput": ["--sem-imputacao"]}

for nome, extra in RESNET_EXTRA.items():
    print("\n" + "=" * 70 + f"\n### {nome}  (ResNet, par de A-xy)\n" + "=" * 70, flush=True)
    subprocess.run([sys.executable, "treinar.py", *BASE_ARGS, *extra,
                    "--saida", str(EXP / nome)], cwd=TREINO, check=True)

for nome, extra in GCN_VARIANTES.items():
    print("\n" + "=" * 70 + f"\n### {nome}  (ST-GCN)\n" + "=" * 70, flush=True)
    subprocess.run([sys.executable, "treinar.py", *GCN_ARGS, *extra,
                    "--saida", str(EXP / nome)], cwd=TREINO, check=True)


## 6. Entregar os artefatos — uso privado


In [ ]:
for rel in sorted(EXP.rglob("relatorio.md")):
    print("=" * 70, "\n", rel.relative_to(EXP))
    print(rel.read_text(encoding="utf-8")[:1500])

artefatos = sorted(p for p in EXP.rglob("*") if p.is_file())
if not artefatos:
    raise RuntimeError("Nenhum artefato para entregar.")
for artefato in artefatos:
    print(artefato.relative_to(EXP), "|", artefato.stat().st_size, "bytes")

# Arquivo fora de EXP para não incluir a si próprio; preserva todos os artefatos.
arquivo = EXP.parent / f"{EXP.name}.tar.gz"
with tarfile.open(arquivo, "w:gz") as tar:
    tar.add(EXP, arcname=EXP.name)
print("\nArquivo completo PRIVADO:", arquivo)

if EM_KAGGLE:
    # Nada a baixar aqui: o download do navegador nem existiria numa execução em
    # background. O .tar.gz está em /kaggle/working e sai como output da versão —
    # aba "Output" da versão, ou "Download all".
    print("Kaggle: pegue o arquivo no Output desta versão (não precisa baixar agora).")
elif EM_COLAB:
    from google.colab import files
    files.download(str(arquivo))
else:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(arquivo, pathlib.Path.cwd())))
